# Passos Mágicos — Análise Exploratória, Treinamento e Avaliação do Modelo

**Objetivo:** Prever o risco de defasagem escolar dos alunos da Associação Passos Mágicos.

**Target:** `risco = 1` se o aluno NÃO está adiantado na série (`Defas >= 0`) · `risco = 0` se está adiantado (`Defas < 0`)

> ⚠️ **Atenção — Data Leakage:** A coluna `IAN` é uma discretização direta de `Defas` e **não deve ser usada como feature**.

## Instalação das libs

In [4]:
import sys
from pathlib import Path

# Garante que o root do projeto está no path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src.preprocessing import (
    ALL_FEATURES, CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET,
    build_preprocessor, clean_data, load_raw_data,
)
from src.feature_engineering import create_features, get_extended_feature_columns
from src.evaluate import evaluate_model, print_summary

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

RANDOM_STATE = 42
TEST_SIZE    = 0.2
CV_FOLDS     = 5

print('Setup OK — root:', ROOT)

Setup OK — root: C:\projetos-pessoais\tech-challenge-5\tech-challenge-5


---
## Carregamento e Visão Geral dos Dados

In [ ]:
df_raw = load_raw_data()
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

2026-03-08 08:35:33,219 [INFO] preprocessing: Loaded base_2024.xlsx: (860, 42)
Shape: (860, 42)


,RA,Fase,Turma,Nome,Ano nasc,Idade 22,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,...,Inglês,Indicado,Atingiu PV,IPV,IAN,Fase ideal,Defas,Destaque IEG,Destaque IDA,Destaque IPV
0,RA-1,7,A,Aluno-1,2003,19,Menina,2016,Escola Pública,Ametista,...,6.0,Sim,Não,7.278,5.0,Fase 8 (Universitários),-1,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
1,RA-2,7,A,Aluno-2,2005,17,Menina,2017,Rede Decisão,Ametista,...,9.7,Não,Não,6.778,10.0,Fase 7 (3º EM),0,Melhorar: Melhorar a sua entrega de lições de ...,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Melhorar: Integrar-se mais aos Princípios Pass...
2,RA-3,7,A,Aluno-3,2005,17,Menina,2016,Rede Decisão,Ametista,...,6.9,Não,Não,7.556,10.0,Fase 7 (3º EM),0,Destaque: A sua boa entrega das lições de casa.,Melhorar: Empenhar-se mais nas aulas e avaliaç...,Destaque: A sua boa integração aos Princípios ...


In [ ]:
df_raw.dtypes.value_counts()

In [ ]:
# Valores ausentes por coluna
missing = df_raw.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f'{len(missing)} colunas com valores ausentes:\n')
print(missing.to_string())

In [ ]:
df_raw.describe(include='all').T

---
## 2. Limpeza e Engenharia de Features

In [ ]:
df = clean_data(df_raw)
df = create_features(df)
print(f'Shape após limpeza + features: {df.shape}')
df.head(3)

In [ ]:
# Colunas novas geradas pela engenharia de features
original_cols = set(clean_data(df_raw).columns)
new_cols = [c for c in df.columns if c not in original_cols]
print('Features derivadas:', new_cols)

---
## 3. Distribuição do Target

In [ ]:
counts = df[TARGET].value_counts().sort_index()
labels = {0: 'Adiantado\n(risco=0)', 1: 'Em risco\n(risco=1)'}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar([labels[k] for k in counts.index], counts.values,
            color=['#4C9BE8', '#E8774C'], edgecolor='white', linewidth=1.2)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')
axes[0].set_title('Distribuição do target (contagem)', fontsize=12)
axes[0].set_ylabel('Quantidade de alunos')

axes[1].pie(counts.values, labels=[labels[k] for k in counts.index],
            autopct='%1.1f%%', colors=['#4C9BE8', '#E8774C'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proporção do target', fontsize=12)

plt.tight_layout()
plt.show()

print(counts.rename(index={0: 'Adiantado (0)', 1: 'Em risco (1)'}).to_string())

---
## 4. Análise Univariada — Variáveis Numéricas

In [ ]:
num_cols = [c for c in NUMERIC_FEATURES if c in df.columns]
n = len(num_cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=25, color='#4C9BE8', edgecolor='white', linewidth=0.6)
    axes[i].set_title(col, fontsize=11)
    axes[i].set_xlabel('')
    med = df[col].median()
    axes[i].axvline(med, color='#E8774C', linestyle='--', linewidth=1.5, label=f'mediana={med:.1f}')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuição das variáveis numéricas', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Estatísticas descritivas
df[num_cols].describe().round(2)

---
## 5. Análise Univariada — Variáveis Categóricas

In [ ]:
cat_cols = [c for c in CATEGORICAL_FEATURES if c in df.columns]

fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 4))
if len(cat_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, cat_cols):
    vc = df[col].value_counts()
    ax.barh(vc.index, vc.values, color='#4C9BE8', edgecolor='white')
    for j, v in enumerate(vc.values):
        ax.text(v + 1, j, str(v), va='center', fontsize=9)
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Contagem')
    ax.invert_yaxis()

plt.suptitle('Distribuição das variáveis categóricas', fontsize=14)
plt.tight_layout()
plt.show()

---
## 6. Análise Bivariada — Features vs Target

In [ ]:
# Boxplots: distribuição de cada feature numérica por classe de risco
ncols = 4
nrows = (len(num_cols) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5))
axes = axes.flatten()

palette = {0: '#4C9BE8', 1: '#E8774C'}
for i, col in enumerate(num_cols):
    data_plot = df[[col, TARGET]].dropna()
    for cls, color in palette.items():
        axes[i].boxplot(
            data_plot.loc[data_plot[TARGET] == cls, col],
            positions=[cls],
            patch_artist=True,
            boxprops=dict(facecolor=color, alpha=0.7),
            medianprops=dict(color='black', linewidth=2),
            widths=0.4,
        )
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(['Adiantado (0)', 'Em risco (1)'], fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribuição das features numéricas por classe de risco', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Taxa de risco por categoria
for col in cat_cols:
    risk_rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 3))
    bars = ax.bar(risk_rate.index, risk_rate.values * 100, color='#E8774C', edgecolor='white')
    for bar, v in zip(bars, risk_rate.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{v:.1%}', ha='center', fontsize=9)
    ax.set_title(f'Taxa de risco (%) por {col}', fontsize=12)
    ax.set_ylabel('% em risco')
    ax.set_ylim(0, 105)
    plt.tight_layout()
    plt.show()

---
## 7. Matriz de Correlação

In [ ]:
extended_cols = get_extended_feature_columns(df)
corr_cols = extended_cols + [TARGET]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.4, ax=ax,
    annot_kws={'size': 7},
)
ax.set_title('Matriz de correlação (Pearson)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlação de cada feature com o target (ordenada)
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#E8774C' if v > 0 else '#4C9BE8' for v in target_corr.values]
ax.barh(target_corr.index[::-1], target_corr.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Correlação das features com "{TARGET}"', fontsize=12)
ax.set_xlabel('Correlação de Pearson')
plt.tight_layout()
plt.show()

---
## 8. Preparação para Treinamento

In [ ]:
feature_cols = [c for c in ALL_FEATURES if c in df.columns]
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f'Features usadas ({len(feature_cols)}): {feature_cols}')
print(f'\nTrain: {len(X_train)} | Test: {len(X_test)}')
print(f'Distribuição no train: {y_train.value_counts().to_dict()}')
print(f'Distribuição no test:  {y_test.value_counts().to_dict()}')

---
## 9. Cross-Validation dos Modelos Candidatos

In [ ]:
candidates = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, random_state=RANDOM_STATE,
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=1,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, clf in candidates.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('classifier', clf)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1', n_jobs=-1)
    cv_results[name] = {'mean': scores.mean(), 'std': scores.std(), 'scores': scores}
    print(f'{name:<25} F1 = {scores.mean():.4f} ± {scores.std():.4f}')

best_name = max(cv_results, key=lambda n: cv_results[n]['mean'])
print(f'\n✓ Melhor modelo (CV F1): {best_name}')

In [ ]:
# Visualização dos resultados de CV
names = list(cv_results.keys())
means = [cv_results[n]['mean'] for n in names]
stds  = [cv_results[n]['std']  for n in names]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#E8774C' if n == best_name else '#4C9BE8' for n in names]
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor='white', linewidth=1.2)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{mean:.4f}', ha='center', fontsize=9, fontweight='bold')
ax.set_title(f'F1 (CV {CV_FOLDS}-fold) por modelo', fontsize=12)
ax.set_ylabel('F1 Score')
ax.set_ylim(max(0, min(means) - 0.05), min(1.05, max(means) + 0.07))
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

---
## 10. Avaliação no Test Set

In [ ]:
fitted_models = {}
test_results = {}

for name, clf in candidates.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    test_results[name] = evaluate_model(pipe, X_test, y_test)

print('=== Comparação no test set ===')
print_summary(test_results)

In [ ]:
# Métricas do melhor modelo em detalhe
best_metrics = test_results[best_name]
print(f'Modelo selecionado: {best_name}\n')
print(classification_report(y_test, fitted_models[best_name].predict(X_test),
                             target_names=['Adiantado (0)', 'Em risco (1)']))

---
## 11. Matriz de Confusão

In [ ]:
ncols = 2
nrows = (len(candidates) + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(11, nrows * 4.5))
axes = axes.flatten()

for i, (name, pipe) in enumerate(fitted_models.items()):
    disp = ConfusionMatrixDisplay.from_estimator(
        pipe, X_test, y_test,
        display_labels=['Adiantado (0)', 'Em risco (1)'],
        cmap='Blues', ax=axes[i], colorbar=False,
    )
    title = f'{name}' + (' ✓ melhor' if name == best_name else '')
    axes[i].set_title(title, fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Matrizes de Confusão — Test Set', fontsize=13)
plt.tight_layout()
plt.show()

---
## 12. Curva ROC

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
colors_roc = ['#4C9BE8', '#E8774C', '#4CAF50', '#9B4CE8']

for (name, pipe), color in zip(fitted_models.items(), colors_roc):
    auc = test_results[name]['roc_auc']
    RocCurveDisplay.from_estimator(
        pipe, X_test, y_test,
        name=f'{name} (AUC={auc:.3f})',
        ax=ax, color=color,
        lw=1.8 if name == best_name else 1.2,
        alpha=1.0 if name == best_name else 0.7,
    )

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Random classifier')
ax.set_title('Curva ROC — Test Set', fontsize=13)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 13. Importância de Features

In [ ]:
# Importância via coeficientes (Logistic Regression) ou feature_importances_ (árvores)
best_pipe = fitted_models[best_name]
clf = best_pipe.named_steps['classifier']
preprocessor = best_pipe.named_steps['preprocessor']

feature_names_out = (
    preprocessor.get_feature_names_out()
    if hasattr(preprocessor, 'get_feature_names_out')
    else feature_cols
)

if hasattr(clf, 'coef_'):
    importances = np.abs(clf.coef_[0])
    kind = 'Coeficiente (|valor|)'
elif hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
    kind = 'Importância (Gini)'
else:
    importances = None

if importances is not None:
    fi = pd.Series(importances, index=feature_names_out).sort_values(ascending=False)
    top_n = min(20, len(fi))

    fig, ax = plt.subplots(figsize=(9, top_n * 0.38 + 1))
    fi_top = fi.head(top_n)
    ax.barh(fi_top.index[::-1], fi_top.values[::-1],
            color='#4C9BE8', edgecolor='white')
    ax.set_title(f'{best_name} — Top {top_n} features ({kind})', fontsize=12)
    ax.set_xlabel(kind)
    plt.tight_layout()
    plt.show()
else:
    print('Modelo não expõe importância de features diretamente.')

---
## 14. Resumo Final

In [ ]:
m = test_results[best_name]
print('=' * 50)
print(f'  Modelo selecionado : {best_name}')
print(f'  CV F1 (train)      : {cv_results[best_name]["mean"]:.4f} ± {cv_results[best_name]["std"]:.4f}')
print(f'  Accuracy (test)    : {m["accuracy"]:.4f}')
print(f'  Precision (test)   : {m["precision"]:.4f}')
print(f'  Recall (test)      : {m["recall"]:.4f}')
print(f'  F1 (test)          : {m["f1"]:.4f}')
print(f'  ROC-AUC (test)     : {m["roc_auc"]:.4f}')
print('=' * 50)